In [ ]:
# Data Imports and Inspector

In [ ]:
from pathlib import Path

import pandas as pd

# Folder holding one exported CSV per listener
DATA_DIR = Path("data")

# Encodings to try in order; utf-8-sig also strips the BOM some exports have
ENCODINGS = ["utf-8-sig", "utf-16", "cp1252", "latin-1"]


def read_csv_any_encoding(path):
    """Read a CSV, trying each encoding until one works."""
    for encoding in ENCODINGS:
        try:
            return pd.read_csv(path, encoding=encoding)
        except (UnicodeError, pd.errors.ParserError):
            # Wrong encoding for this file, try the next one
            continue
    raise ValueError(f"Could not read {path} with any of {ENCODINGS}")


# Load every CSV in the data folder and tag its rows with the listener
frames = []
for path in sorted(DATA_DIR.glob("*.csv")):
    df = read_csv_any_encoding(path)
    # Listener name is the first word of the file name,
    # e.g. "Logan Exported Data.csv" -> "Logan"
    df["Listener"] = path.stem.split()[0]
    frames.append(df)

# Stack all listeners into one DataFrame
data = pd.concat(frames, ignore_index=True)

# Quick sanity check: total size and rows per listener
print(data.shape)
print(data["Listener"].value_counts())
data.head()